<a href="https://colab.research.google.com/github/samuraiinst2025/test/blob/main/test/20260203_SHsama.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 新しいセクション

In [2]:
#test
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from pathlib import Path
import re
from openpyxl import load_workbook, Workbook

# ===============================
# パス設定
# ===============================
INPUT_DIR = Path("/content/order")
RESULT_DIR = INPUT_DIR / "result"
RESULT_DIR.mkdir(parents=True, exist_ok=True)

OUT_PATH = RESULT_DIR / "order_Total_20230524.xlsx"

# ===============================
# 出力列定義（固定）
# ===============================
VEG_COL = {
    "トマト": 2,
    "キャベツ": 3,
    "レタス": 4,
    "白菜": 5,
    "ほうれん草": 6,
    "大根": 7,
    "ニンジン": 8,   # H列
}

HEADERS = [
    "ソース名",
    "トマト",
    "キャベツ",
    "レタス",
    "白菜",
    "ほうれん草",
    "大根",
    "ニンジン",
]

# ===============================
# ユーティリティ
# ===============================
def normalize_header(v):
    """見出しの表記ゆれ対策"""
    if v is None:
        return ""
    s = str(v).strip()
    s = s.replace("　", "")                 # 全角スペース除去
    s = re.sub(r"\(.*?\)$", "", s)          # (kg) 等除去
    return s

def safe_float(v):
    """数値 or 数値っぽい文字列を float に"""
    if v is None:
        return None
    if isinstance(v, (int, float)):
        return float(v)
    if isinstance(v, str):
        s = re.sub(r"[^0-9\.\-]", "", v)
        try:
            return float(s)
        except:
            return None
    return None

# ===============================
# 対象ファイル取得
# ===============================
targets = [
    f for f in sorted(INPUT_DIR.glob("*.xlsx"))
    if not f.name.startswith("~$")
]

if not targets:
    raise FileNotFoundError(f"処理対象の Excel ファイルがありません: {INPUT_DIR}")

print("対象ファイル:")
for f in targets:
    print(" -", f.name)

# ===============================
# ① データ抽出（横並び）
# ===============================
records = []

for fp in targets:
    wb = load_workbook(fp, data_only=True)
    ws = wb.worksheets[0]   # 先頭シート

    rec = {"source": fp.name}

    for c in range(1, ws.max_column + 1):
        header = normalize_header(ws.cell(row=1, column=c).value)
        if header in VEG_COL:
            qty = safe_float(ws.cell(row=2, column=c).value)
            rec[header] = qty

    records.append(rec)
    wb.close()  # 明示クローズ

# ===============================
# ② 集計ファイル作成（毎回新規）
# ===============================
out_wb = Workbook()
out_ws = out_wb.active
out_ws.title = "Total"

# ヘッダ
for c, h in enumerate(HEADERS, start=1):
    out_ws.cell(row=1, column=c, value=h)

# データ行
for r, rec in enumerate(records, start=2):
    out_ws.cell(row=r, column=1, value=rec["source"])
    for veg, col in VEG_COL.items():
        out_ws.cell(row=r, column=col, value=rec.get(veg))

out_wb.save(OUT_PATH)
out_wb.close()

# ===============================
# ③ TOTAL 行（列合計）
# ===============================
sum_wb = load_workbook(OUT_PATH)
ws = sum_wb["Total"]

last_row = ws.max_row
last_col = ws.max_column

ws.cell(row=last_row + 1, column=1, value="TOTAL")

for c in range(2, last_col + 1):
    total = 0.0
    has_value = False
    for r in range(2, last_row + 1):
        v = safe_float(ws.cell(row=r, column=c).value)
        if v is not None:
            total += v
            has_value = True
    if has_value:
        ws.cell(row=last_row + 1, column=c, value=total)

sum_wb.save(OUT_PATH)
sum_wb.close()

print("完了しました。出力先:", OUT_PATH)


対象ファイル:
 - order_A_20230524.xlsx
 - order_B_20230524.xlsx
 - order_C_20230524.xlsx
 - order_D_20230524.xlsx
完了しました。出力先: /content/order/result/order_Total_20230524.xlsx


In [ ]:
from pathlib import Path
from openpyxl import load_workbook, Workbook
import re

# =========================
# パス設定（確定）
# =========================
ORDER_PATH = Path("/content/order/result/order_Total_20230524.xlsx")
INVENTORY_PATH = Path("/content/inventory/inventory.xlsx")

OUT_DIR = Path("/content/inventory/result")
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_PATH = OUT_DIR / "inventory_analize_20230524.xlsx"

# =========================
# 存在チェック
# =========================
if not ORDER_PATH.exists():
    raise FileNotFoundError(f"order_Total_20230524.xlsx が見つかりません: {ORDER_PATH}")

if not INVENTORY_PATH.exists():
    raise FileNotFoundError(f"inventory.xlsx が見つかりません: {INVENTORY_PATH}")

print("order file:", ORDER_PATH)
print("inventory file:", INVENTORY_PATH)
print("output file:", OUT_PATH)

# =========================
# ユーティリティ
# =========================
def normalize_header(v):
    if v is None:
        return ""
    s = str(v).strip().replace("　", "")
    s = re.sub(r"\(.*?\)$", "", s)
    return s

def to_number(v):
    if v is None:
        return None
    if isinstance(v, (int, float)):
        return float(v)
    if isinstance(v, str):
        s = re.sub(r"[^0-9\.\-]", "", v.strip())
        if s in ("", "-", ".", "-."):
            return None
        try:
            return float(s)
        except:
            return None
    return None

def get_row(ws, row_idx, max_col):
    return [ws.cell(row=row_idx, column=c).value for c in range(1, max_col + 1)]

# =========================
# ① order_Total の 1行目・最終行目
# =========================
order_wb = load_workbook(ORDER_PATH, data_only=True)
order_ws = order_wb.worksheets[0]

order_header = get_row(order_ws, 1, order_ws.max_column)
order_last = get_row(order_ws, order_ws.max_row, order_ws.max_column)

order_wb.close()

# =========================
# ② inventory の 最終行目
# =========================
inv_wb = load_workbook(INVENTORY_PATH, data_only=True)
inv_ws = inv_wb.worksheets[0]

inv_header = get_row(inv_ws, 1, inv_ws.max_column)
inv_last = get_row(inv_ws, inv_ws.max_row, inv_ws.max_column)

inv_wb.close()

# inventory を「列名 → 値」に変換
inv_map = {}
for h, v in zip(inv_header, inv_last):
    key = normalize_header(h)
    if key:
        inv_map[key] = v

# =========================
# ③ 新規ファイル作成・①②を書き出し
# =========================
out_wb = Workbook()
out_ws = out_wb.active
out_ws.title = "Analyze"

# 1行目：order のヘッダ
for c, v in enumerate(order_header, start=1):
    out_ws.cell(row=1, column=c, value=v)

# 2行目：order の最終行（TOTAL）
out_ws.cell(row=2, column=1, value="order_total_last")
for c in range(2, len(order_header) + 1):
    out_ws.cell(row=2, column=c, value=order_last[c - 1])

# 3行目：inventory の最終行（order ヘッダ順）
out_ws.cell(row=3, column=1, value="inventory_last")
for c in range(2, len(order_header) + 1):
    col_name = normalize_header(order_header[c - 1])
    out_ws.cell(row=3, column=c, value=inv_map.get(col_name))

# =========================
# ④ 差分（3行目 − 2行目）を 4行目に追記
# =========================
out_ws.cell(row=4, column=1, value="DIFF(3-2)")
for c in range(2, len(order_header) + 1):
    v2 = to_number(out_ws.cell(row=2, column=c).value)
    v3 = to_number(out_ws.cell(row=3, column=c).value)
    out_ws.cell(row=4, column=c, value=(v3 - v2) if v2 is not None and v3 is not None else None)

# 保存してクローズ
out_wb.save(OUT_PATH)
out_wb.close()

print("完了しました。出力:", OUT_PATH)


order file: /content/order/result/order_Total_20230524.xlsx
inventory file: /content/inventory/inventory.xlsx
output file: /content/inventory/result/inventory_analize_20230524.xlsx
完了しました。出力: /content/inventory/result/inventory_analize_20230524.xlsx


In [ ]:
from pathlib import Path
from openpyxl import load_workbook, Workbook
import re

# =====================================
# パス設定（確定）
# =====================================
ANALYZE_PATH = Path("/content/inventory/result/inventory_analize_20230524.xlsx")
PICKUP_PATH  = Path("/content/inventory/pickup.xlsx")

OUT_DIR  = Path("/content/inventory/result")
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_PATH = OUT_DIR / "inventory_pickup_20230524.xlsx"

# =====================================
# 存在チェック
# =====================================
if not ANALYZE_PATH.exists():
    raise FileNotFoundError(f"inventory_analize_20230524.xlsx が見つかりません: {ANALYZE_PATH}")

if not PICKUP_PATH.exists():
    raise FileNotFoundError(f"pickup.xlsx が見つかりません: {PICKUP_PATH}")

print("analyze file:", ANALYZE_PATH)
print("pickup file :", PICKUP_PATH)
print("output file :", OUT_PATH)

# =====================================
# ユーティリティ
# =====================================
def to_number(v):
    """数値 or 数値っぽい文字列を float に変換（不可なら None）"""
    if v is None:
        return None
    if isinstance(v, (int, float)):
        return float(v)
    if isinstance(v, str):
        s = re.sub(r"[^0-9\.\-]", "", v.strip())
        if s in ("", "-", ".", "-."):
            return None
        try:
            return float(s)
        except:
            return None
    return None

def get_row(ws, row_idx, max_col=None):
    if max_col is None:
        max_col = ws.max_column
    return [ws.cell(row=row_idx, column=c).value for c in range(1, max_col + 1)]

def pad(row, n):
    return row + [None] * (n - len(row)) if len(row) < n else row[:n]

# =====================================
# ① inventory_analize 最終行 + ヘッダ取得
# =====================================
an_wb = load_workbook(ANALYZE_PATH, data_only=True)
an_ws = an_wb.worksheets[0]

an_header = get_row(an_ws, 1, an_ws.max_column)
an_last   = get_row(an_ws, an_ws.max_row, an_ws.max_column)

an_wb.close()

# =====================================
# ② pickup.xlsx 2行目・3行目取得
# =====================================
pk_wb = load_workbook(PICKUP_PATH, data_only=True)
pk_ws = pk_wb.worksheets[0]

pk_row2 = get_row(pk_ws, 2, pk_ws.max_column)
pk_row3 = get_row(pk_ws, 3, pk_ws.max_column)

pk_wb.close()

# =====================================
# 列数を揃える
# =====================================
max_cols = max(len(an_header), len(an_last), len(pk_row2), len(pk_row3))
an_header = pad(an_header, max_cols)
an_last   = pad(an_last, max_cols)
pk_row2   = pad(pk_row2, max_cols)
pk_row3   = pad(pk_row3, max_cols)

# =====================================
# ③ 新規ファイル作成・情報追記
# =====================================
out_wb = Workbook()
out_ws = out_wb.active
out_ws.title = "Pickup"

# 1行目：ヘッダ
for c, v in enumerate(an_header, start=1):
    out_ws.cell(row=1, column=c, value=v)

# 2行目：inventory_analize 最終行
for c, v in enumerate(an_last, start=1):
    out_ws.cell(row=2, column=c, value=v)

# 3行目：pickup 2行目
for c, v in enumerate(pk_row2, start=1):
    out_ws.cell(row=3, column=c, value=v)

# 4行目：pickup 3行目
for c, v in enumerate(pk_row3, start=1):
    out_ws.cell(row=4, column=c, value=v)

out_wb.save(OUT_PATH)
out_wb.close()
print("③ ファイル作成完了:", OUT_PATH)

# =====================================
# ④ 差分（2行目 − 3行目）を5行目に追記
#     ＋ 指定セル文言を更新して保存
# =====================================
wb = load_workbook(OUT_PATH)
ws = wb["Pickup"]

# 差分行
ws.cell(row=5, column=1, value="DIFF(2-3)")
for c in range(2, max_cols + 1):
    v2 = to_number(ws.cell(row=2, column=c).value)
    v3 = to_number(ws.cell(row=3, column=c).value)
    ws.cell(
        row=5,
        column=c,
        value=(v2 - v3) if v2 is not None and v3 is not None else None
    )

# ★ 指定どおりラベルを更新
ws["A2"] = "最新在庫数量"
ws["A5"] = "発注判定"

# 保存してクローズ
wb.save(OUT_PATH)
wb.close()

print("④ ラベル更新・差分計算完了。最終出力:", OUT_PATH)


analyze file: /content/inventory/result/inventory_analize_20230524.xlsx
pickup file : /content/inventory/pickup.xlsx
output file : /content/inventory/result/inventory_pickup_20230524.xlsx
③ ファイル作成完了: /content/inventory/result/inventory_pickup_20230524.xlsx
④ ラベル更新・差分計算完了。最終出力: /content/inventory/result/inventory_pickup_20230524.xlsx


In [ ]:
from pathlib import Path
from openpyxl import load_workbook
import re
from datetime import datetime

FILE_PATH = Path("/content/inventory/result/inventory_pickup_20230524.xlsx")

def to_number(v):
    """数値 or 数値っぽい文字列 -> float / 不可なら None"""
    if v is None:
        return None
    if isinstance(v, (int, float)):
        return float(v)
    if isinstance(v, str):
        s = re.sub(r"[^0-9\.\-]", "", v.strip())
        if s in ("", "-", ".", "-."):
            return None
        try:
            return float(s)
        except:
            return None
    return None

def read_row_map(ws, label_text):
    """
    A列(1列目)に label_text がある行を探し、
    その行の {header: value} を返す
    """
    max_row = ws.max_row
    max_col = ws.max_column

    # 1行目はヘッダ（野菜名など）として扱う
    headers = [ws.cell(row=1, column=c).value for c in range(1, max_col + 1)]

    target_row = None
    for r in range(2, max_row + 1):
        a = ws.cell(row=r, column=1).value
        if a == label_text:
            target_row = r
            break

    if target_row is None:
        raise ValueError(f"行ラベル '{label_text}' がA列に見つかりません。")

    row_values = [ws.cell(row=target_row, column=c).value for c in range(1, max_col + 1)]

    # A列はラベルなので除外し、B列以降を {header: value} へ
    data = {}
    for c in range(2, max_col + 1):
        key = headers[c - 1]
        if key is None or str(key).strip() == "":
            continue
        data[str(key).strip()] = row_values[c - 1]
    return data

def build_order_list(xlsx_path: Path):
    if not xlsx_path.exists():
        raise FileNotFoundError(f"ファイルが見つかりません: {xlsx_path}")

    wb = load_workbook(xlsx_path, data_only=True)
    ws = wb.worksheets[0]

    judgement_map = read_row_map(ws, "発注判定")   # 野菜 -> 判定値
    addqty_map    = read_row_map(ws, "追加量")     # 野菜 -> 追加量

    wb.close()

    order_items = []
    for veg, jv in judgement_map.items():
        j = to_number(jv)
        if j is None:
            continue
        if j <= 0:
            qty_raw = addqty_map.get(veg)
            qty = to_number(qty_raw)
            if qty is None:
                # 追加量が空欄/非数値だと誤発注リスクがあるので止める設計
                raise ValueError(f"'{veg}' の追加量が数値として取得できません（追加量セル={qty_raw!r}）。")
            order_items.append((veg, qty))

    return order_items

def render_email(subject_date: str, supplier_name: str, order_items: list[tuple[str, float]]):
    """
    order_items: [(野菜名, 追加量), ...]
    """
    # 本文の明細行（見やすく固定幅風）
    lines = []
    for veg, qty in order_items:
        # qty が小数の可能性もあるため、そのまま出す（必要なら丸め）
        lines.append(f"・{veg}：{qty:g}")

    items_block = "\n".join(lines) if lines else "（発注対象なし）"

    subject = f"【発注依頼】{subject_date}分 野菜の追加発注"
    body = f"""\
{supplier_name} ご担当者様

お世話になっております。
在庫状況を確認したところ、下記の通り追加発注をお願いいたします。

■発注内容（追加量）
{items_block}

■納品希望
・可能であれば最短便でお願いいたします。
・納品日が難しい場合は、最短の納品予定日をご返信ください。

以上、よろしくお願いいたします。
"""
    return subject, body

# ===== 実行 =====
order_items = build_order_list(FILE_PATH)

# 例：サプライヤー名（固定でOK）
SUPPLIER_NAME = "〇〇農園"

today = datetime.now().strftime("%Y/%m/%d")
subject, body = render_email(today, SUPPLIER_NAME, order_items)

print("==== 発注対象 ====")
if order_items:
    for veg, qty in order_items:
        print(f"{veg}: {qty:g}")
else:
    print("発注対象なし")

print("\n==== 件名 ====")
print(subject)
print("\n==== 本文 ====")
print(body)


==== 発注対象 ====
ニンジン: 80

==== 件名 ====
【発注依頼】2026/02/03分 野菜の追加発注

==== 本文 ====
〇〇農園 ご担当者様

お世話になっております。
在庫状況を確認したところ、下記の通り追加発注をお願いいたします。

■発注内容（追加量）
・ニンジン：80

■納品希望
・可能であれば最短便でお願いいたします。
・納品日が難しい場合は、最短の納品予定日をご返信ください。

以上、よろしくお願いいたします。



In [ ]:
from pathlib import Path
import re
from datetime import datetime
from openpyxl import load_workbook, Workbook

# =========================================================
# 共通ユーティリティ（重複排除）
# =========================================================
def normalize_header(v):
    """見出しの表記ゆれ対策"""
    if v is None:
        return ""
    s = str(v).strip()
    s = s.replace("　", "")                 # 全角スペース除去
    s = re.sub(r"\(.*?\)$", "", s)          # (kg) 等除去
    return s

def to_number(v):
    """数値 or 数値っぽい文字列 -> float / 不可なら None"""
    if v is None:
        return None
    if isinstance(v, (int, float)):
        return float(v)
    if isinstance(v, str):
        s = re.sub(r"[^0-9\.\-]", "", v.strip())
        if s in ("", "-", ".", "-."):
            return None
        try:
            return float(s)
        except:
            return None
    return None

def safe_float(v):
    """（Part1互換）数値 or 数値っぽい文字列 -> float"""
    return to_number(v)

def get_row(ws, row_idx, max_col):
    return [ws.cell(row=row_idx, column=c).value for c in range(1, max_col + 1)]

def pad(row, n):
    return row + [None] * (n - len(row)) if len(row) < n else row[:n]


# =========================================================
# Part1: order_Total_20230524.xlsx 作成
# =========================================================
def run_part1_make_order_total():
    INPUT_DIR = Path("/content/order")
    RESULT_DIR = INPUT_DIR / "result"
    RESULT_DIR.mkdir(parents=True, exist_ok=True)
    OUT_PATH = RESULT_DIR / "order_Total_20230524.xlsx"

    VEG_COL = {
        "トマト": 2,
        "キャベツ": 3,
        "レタス": 4,
        "白菜": 5,
        "ほうれん草": 6,
        "大根": 7,
        "ニンジン": 8,   # H列
    }

    HEADERS = [
        "ソース名",
        "トマト",
        "キャベツ",
        "レタス",
        "白菜",
        "ほうれん草",
        "大根",
        "ニンジン",
    ]

    targets = [
        f for f in sorted(INPUT_DIR.glob("*.xlsx"))
        if not f.name.startswith("~$")
    ]
    if not targets:
        raise FileNotFoundError(f"処理対象の Excel ファイルがありません: {INPUT_DIR}")

    print("[Part1] 対象ファイル:")
    for f in targets:
        print(" -", f.name)

    records = []
    for fp in targets:
        wb = load_workbook(fp, data_only=True)
        ws = wb.worksheets[0]

        rec = {"source": fp.name}
        for c in range(1, ws.max_column + 1):
            header = normalize_header(ws.cell(row=1, column=c).value)
            if header in VEG_COL:
                qty = safe_float(ws.cell(row=2, column=c).value)
                rec[header] = qty

        records.append(rec)
        wb.close()

    out_wb = Workbook()
    out_ws = out_wb.active
    out_ws.title = "Total"

    for c, h in enumerate(HEADERS, start=1):
        out_ws.cell(row=1, column=c, value=h)

    for r, rec in enumerate(records, start=2):
        out_ws.cell(row=r, column=1, value=rec["source"])
        for veg, col in VEG_COL.items():
            out_ws.cell(row=r, column=col, value=rec.get(veg))

    out_wb.save(OUT_PATH)
    out_wb.close()

    sum_wb = load_workbook(OUT_PATH)
    ws = sum_wb["Total"]

    last_row = ws.max_row
    last_col = ws.max_column

    ws.cell(row=last_row + 1, column=1, value="TOTAL")

    for c in range(2, last_col + 1):
        total = 0.0
        has_value = False
        for r in range(2, last_row + 1):
            v = safe_float(ws.cell(row=r, column=c).value)
            if v is not None:
                total += v
                has_value = True
        if has_value:
            ws.cell(row=last_row + 1, column=c, value=total)

    sum_wb.save(OUT_PATH)
    sum_wb.close()

    print("[Part1] 完了。出力先:", OUT_PATH)
    return OUT_PATH


# =========================================================
# Part2: inventory_analize_20230524.xlsx 作成
# =========================================================
def run_part2_make_inventory_analyze():
    ORDER_PATH = Path("/content/order/result/order_Total_20230524.xlsx")
    INVENTORY_PATH = Path("/content/inventory/inventory.xlsx")

    OUT_DIR = Path("/content/inventory/result")
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    OUT_PATH = OUT_DIR / "inventory_analize_20230524.xlsx"

    if not ORDER_PATH.exists():
        raise FileNotFoundError(f"order_Total_20230524.xlsx が見つかりません: {ORDER_PATH}")
    if not INVENTORY_PATH.exists():
        raise FileNotFoundError(f"inventory.xlsx が見つかりません: {INVENTORY_PATH}")

    print("[Part2] order file    :", ORDER_PATH)
    print("[Part2] inventory file:", INVENTORY_PATH)

    order_wb = load_workbook(ORDER_PATH, data_only=True)
    order_ws = order_wb.worksheets[0]
    order_header = get_row(order_ws, 1, order_ws.max_column)
    order_last = get_row(order_ws, order_ws.max_row, order_ws.max_column)
    order_wb.close()

    inv_wb = load_workbook(INVENTORY_PATH, data_only=True)
    inv_ws = inv_wb.worksheets[0]
    inv_header = get_row(inv_ws, 1, inv_ws.max_column)
    inv_last = get_row(inv_ws, inv_ws.max_row, inv_ws.max_column)
    inv_wb.close()

    inv_map = {}
    for h, v in zip(inv_header, inv_last):
        key = normalize_header(h)
        if key:
            inv_map[key] = v

    out_wb = Workbook()
    out_ws = out_wb.active
    out_ws.title = "Analyze"

    for c, v in enumerate(order_header, start=1):
        out_ws.cell(row=1, column=c, value=v)

    out_ws.cell(row=2, column=1, value="order_total_last")
    for c in range(2, len(order_header) + 1):
        out_ws.cell(row=2, column=c, value=order_last[c - 1])

    out_ws.cell(row=3, column=1, value="inventory_last")
    for c in range(2, len(order_header) + 1):
        col_name = normalize_header(order_header[c - 1])
        out_ws.cell(row=3, column=c, value=inv_map.get(col_name))

    out_ws.cell(row=4, column=1, value="DIFF(3-2)")
    for c in range(2, len(order_header) + 1):
        v2 = to_number(out_ws.cell(row=2, column=c).value)
        v3 = to_number(out_ws.cell(row=3, column=c).value)
        out_ws.cell(row=4, column=c, value=(v3 - v2) if v2 is not None and v3 is not None else None)

    out_wb.save(OUT_PATH)
    out_wb.close()

    print("[Part2] 完了。出力先:", OUT_PATH)
    return OUT_PATH


# =========================================================
# Part3: inventory_pickup_20230524.xlsx 作成（A2/A5ラベル更新込み）
# =========================================================
def run_part3_make_inventory_pickup():
    ANALYZE_PATH = Path("/content/inventory/result/inventory_analize_20230524.xlsx")
    PICKUP_PATH  = Path("/content/inventory/pickup.xlsx")

    OUT_DIR  = Path("/content/inventory/result")
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    OUT_PATH = OUT_DIR / "inventory_pickup_20230524.xlsx"

    if not ANALYZE_PATH.exists():
        raise FileNotFoundError(f"inventory_analize_20230524.xlsx が見つかりません: {ANALYZE_PATH}")
    if not PICKUP_PATH.exists():
        raise FileNotFoundError(f"pickup.xlsx が見つかりません: {PICKUP_PATH}")

    print("[Part3] analyze file:", ANALYZE_PATH)
    print("[Part3] pickup file :", PICKUP_PATH)

    an_wb = load_workbook(ANALYZE_PATH, data_only=True)
    an_ws = an_wb.worksheets[0]
    an_header = get_row(an_ws, 1, an_ws.max_column)
    an_last   = get_row(an_ws, an_ws.max_row, an_ws.max_column)
    an_wb.close()

    pk_wb = load_workbook(PICKUP_PATH, data_only=True)
    pk_ws = pk_wb.worksheets[0]
    pk_row2 = get_row(pk_ws, 2, pk_ws.max_column)
    pk_row3 = get_row(pk_ws, 3, pk_ws.max_column)
    pk_wb.close()

    max_cols = max(len(an_header), len(an_last), len(pk_row2), len(pk_row3))
    an_header = pad(an_header, max_cols)
    an_last   = pad(an_last, max_cols)
    pk_row2   = pad(pk_row2, max_cols)
    pk_row3   = pad(pk_row3, max_cols)

    out_wb = Workbook()
    out_ws = out_wb.active
    out_ws.title = "Pickup"

    for c, v in enumerate(an_header, start=1):
        out_ws.cell(row=1, column=c, value=v)
    for c, v in enumerate(an_last, start=1):
        out_ws.cell(row=2, column=c, value=v)
    for c, v in enumerate(pk_row2, start=1):
        out_ws.cell(row=3, column=c, value=v)
    for c, v in enumerate(pk_row3, start=1):
        out_ws.cell(row=4, column=c, value=v)

    out_wb.save(OUT_PATH)
    out_wb.close()
    print("[Part3] ③ ファイル作成完了:", OUT_PATH)

    wb = load_workbook(OUT_PATH)
    ws = wb["Pickup"]

    ws.cell(row=5, column=1, value="DIFF(2-3)")
    for c in range(2, max_cols + 1):
        v2 = to_number(ws.cell(row=2, column=c).value)
        v3 = to_number(ws.cell(row=3, column=c).value)
        ws.cell(row=5, column=c, value=(v2 - v3) if v2 is not None and v3 is not None else None)

    ws["A2"] = "最新在庫数量"
    ws["A5"] = "発注判定"

    wb.save(OUT_PATH)
    wb.close()

    print("[Part3] ④ ラベル更新・差分計算完了。出力先:", OUT_PATH)
    return OUT_PATH


# =========================================================
# Part4: 発注対象抽出 + メール文面生成
# =========================================================
from pathlib import Path
from openpyxl import load_workbook
import re
from datetime import datetime
import smtplib
from email.message import EmailMessage

# =========================
# Mailtrap（直書き）
# =========================
username = '**************'
password = '**************'
sender_address = 'from@example.com'
receiver_address = 'to@example.com'

# Mailtrap SMTP設定（Mailtrap画面の値に合わせてください）
smtp_host = "smtp.mailtrap.io"     # 例：smtp.mailtrap.io / sandbox.smtp.mailtrap.io 等
smtp_port = 587                   # 例：587(TLS) / 2525 / 25 など

# =========================
# 対象ファイル
# =========================
FILE_PATH = Path("/content/inventory/result/inventory_pickup_20230524.xlsx")

# =========================
# ユーティリティ
# =========================
def to_number(v):
    if v is None:
        return None
    if isinstance(v, (int, float)):
        return float(v)
    if isinstance(v, str):
        s = re.sub(r"[^0-9\.\-]", "", v.strip())
        if s in ("", "-", ".", "-."):
            return None
        try:
            return float(s)
        except:
            return None
    return None

def norm_label(x):
    # 前後空白・全角スペースの揺れを吸収
    if x is None:
        return ""
    return str(x).strip().replace("　", "")

def read_row_map(ws, label_text):
    """
    A列に label_text がある行を探し、
    その行の {ヘッダ: 値} を返す（B列以降）
    """
    max_row = ws.max_row
    max_col = ws.max_column
    headers = [ws.cell(row=1, column=c).value for c in range(1, max_col + 1)]

    target_row = None
    for r in range(2, max_row + 1):
        a = ws.cell(row=r, column=1).value
        if norm_label(a) == norm_label(label_text):
            target_row = r
            break

    if target_row is None:
        raise ValueError(f"行ラベル '{label_text}' がA列に見つかりません。")

    row_values = [ws.cell(row=target_row, column=c).value for c in range(1, max_col + 1)]

    data = {}
    for c in range(2, max_col + 1):
        key = headers[c - 1]
        if key is None or str(key).strip() == "":
            continue
        data[str(key).strip()] = row_values[c - 1]
    return data

def build_order_list(xlsx_path: Path):
    if not xlsx_path.exists():
        raise FileNotFoundError(f"ファイルが見つかりません: {xlsx_path}")

    wb = load_workbook(xlsx_path, data_only=True)
    ws = wb.worksheets[0]

    # 発注判定（<=0の野菜を抽出）
    judgement_map = read_row_map(ws, "発注判定")

    # 追加量（発注数量）
    addqty_map = read_row_map(ws, "追加量")

    wb.close()

    order_items = []
    for veg, jv in judgement_map.items():
        j = to_number(jv)
        if j is None:
            continue
        if j <= 0:
            qty_raw = addqty_map.get(veg)
            qty = to_number(qty_raw)
            if qty is None:
                raise ValueError(f"'{veg}' の追加量が数値として取得できません（追加量セル={qty_raw!r}）。")
            # 追加量が0なら除外したい場合は、次の2行を有効化
            # if qty <= 0:
            #     continue
            order_items.append((veg, qty))

    return order_items

def render_email(subject_date: str, supplier_name: str, order_items):
    lines = [f"・{veg}：{qty:g}" for veg, qty in order_items]
    items_block = "\n".join(lines) if lines else "（発注対象なし）"

    subject = f"【発注依頼】{subject_date}分 野菜の追加発注"
    body = f"""{supplier_name} ご担当者様

お世話になっております。
在庫状況を確認したところ、下記の通り追加発注をお願いいたします。

■発注内容（追加量）
{items_block}

■納品希望
・可能であれば最短便でお願いいたします。
・納品日が難しい場合は、最短の納品予定日をご返信ください。

以上、よろしくお願いいたします。
"""
    return subject, body

def send_via_mailtrap(host, port, username, password, sender, receiver, subject, body):
    msg = EmailMessage()
    msg["From"] = sender
    msg["To"] = receiver
    msg["Subject"] = subject
    msg.set_content(body)

    with smtplib.SMTP(host, port) as server:
        server.ehlo()
        # TLSが必要な設定が多い（Mailtrapの画面に従う）
        try:
            server.starttls()
            server.ehlo()
        except smtplib.SMTPException:
            # TLSなしポートの場合もあるので握りつぶし（必要なら厳格化）
            pass

        server.login(username, password)
        server.send_message(msg)

# =========================
# 実行
# =========================
order_items = build_order_list(FILE_PATH)

supplier_name = "〇〇農園"
today = datetime.now().strftime("%Y/%m/%d")
subject, body = render_email(today, supplier_name, order_items)

print("==== 発注対象 ====")
if order_items:
    for veg, qty in order_items:
        print(f"{veg}: {qty:g}")
else:
    print("発注対象なし")

print("\n==== 件名 ====")
print(subject)
print("\n==== 本文 ====")
print(body)

# 発注対象なしなら送信しない（事故防止）
if not order_items:
    print("\n送信スキップ：発注対象なし")
else:
    send_via_mailtrap(
        host=smtp_host,
        port=smtp_port,
        username=username,
        password=password,
        sender=sender_address,
        receiver=receiver_address,
        subject=subject,
        body=body,
    )
    print("\n送信完了（Mailtrapに投函）")



# =========================================================
# main: 順番実行
# =========================================================
def main():
    print("=== Pipeline Start ===")
    out1 = run_part1_make_order_total()
    out2 = run_part2_make_inventory_analyze()
    out3 = run_part3_make_inventory_pickup()
    out4 = run_part4_make_order_email()
    print("=== Pipeline Done ===")

    return {
        "order_total": out1,
        "inventory_analyze": out2,
        "inventory_pickup": out3,
        "email": out4,
    }

# 実行
results = main()
results


=== Pipeline Start ===
[Part1] 対象ファイル:
 - order_A_20230524.xlsx
 - order_B_20230524.xlsx
 - order_C_20230524.xlsx
 - order_D_20230524.xlsx
[Part1] 完了。出力先: /content/order/result/order_Total_20230524.xlsx
[Part2] order file    : /content/order/result/order_Total_20230524.xlsx
[Part2] inventory file: /content/inventory/inventory.xlsx
[Part2] 完了。出力先: /content/inventory/result/inventory_analize_20230524.xlsx
[Part3] analyze file: /content/inventory/result/inventory_analize_20230524.xlsx
[Part3] pickup file : /content/inventory/pickup.xlsx
[Part3] ③ ファイル作成完了: /content/inventory/result/inventory_pickup_20230524.xlsx
[Part3] ④ ラベル更新・差分計算完了。出力先: /content/inventory/result/inventory_pickup_20230524.xlsx
[Part4] ==== 発注対象 ====
ニンジン: 80

[Part4] ==== 件名 ====
【発注依頼】2026/02/03分 野菜の追加発注

[Part4] ==== 本文 ====
〇〇農園 ご担当者様

お世話になっております。
在庫状況を確認したところ、下記の通り追加発注をお願いいたします。

■発注内容（追加量）
・ニンジン：80

■納品希望
・可能であれば最短便でお願いいたします。
・納品日が難しい場合は、最短の納品予定日をご返信ください。

以上、よろしくお願いいたします。

=== Pipeline Done ===


{'order_total': PosixPath('/content/order/result/order_Total_20230524.xlsx'),
 'inventory_analyze': PosixPath('/content/inventory/result/inventory_analize_20230524.xlsx'),
 'inventory_pickup': PosixPath('/content/inventory/result/inventory_pickup_20230524.xlsx'),
 'email': {'order_items': [('ニンジン', 80.0)],
  'subject': '【発注依頼】2026/02/03分 野菜の追加発注',
  'body': '〇〇農園 ご担当者様\n\nお世話になっております。\n在庫状況を確認したところ、下記の通り追加発注をお願いいたします。\n\n■発注内容（追加量）\n・ニンジン：80\n\n■納品希望\n・可能であれば最短便でお願いいたします。\n・納品日が難しい場合は、最短の納品予定日をご返信ください。\n\n以上、よろしくお願いいたします。\n'}}

In [ ]:
from pathlib import Path
import re
from datetime import datetime
from openpyxl import load_workbook, Workbook
import smtplib
from email.message import EmailMessage

# =========================================================
# 共通ユーティリティ
# =========================================================
def normalize_header(v):
    """見出しの表記ゆれ対策"""
    if v is None:
        return ""
    s = str(v).strip()
    s = s.replace("　", "")                 # 全角スペース除去
    s = re.sub(r"\(.*?\)$", "", s)          # (kg) 等除去
    return s

def to_number(v):
    """数値 or 数値っぽい文字列 -> float / 不可なら None"""
    if v is None:
        return None
    if isinstance(v, (int, float)):
        return float(v)
    if isinstance(v, str):
        s = re.sub(r"[^0-9\.\-]", "", v.strip())
        if s in ("", "-", ".", "-."):
            return None
        try:
            return float(s)
        except:
            return None
    return None

def safe_float(v):
    """（Part1互換）数値 or 数値っぽい文字列を float に"""
    return to_number(v)

def get_row(ws, row_idx, max_col):
    return [ws.cell(row=row_idx, column=c).value for c in range(1, max_col + 1)]

def pad(row, n):
    return row + [None] * (n - len(row)) if len(row) < n else row[:n]

def norm_label(x):
    """A列ラベル照合用：前後空白・全角スペース除去"""
    if x is None:
        return ""
    return str(x).strip().replace("　", "")


# =========================================================
# Part1: order_Total_20230524.xlsx 作成
#   入力: /content/order/*.xlsx
#   出力: /content/order/result/order_Total_20230524.xlsx
# =========================================================
def run_part1_make_order_total():
    INPUT_DIR = Path("/content/order")
    RESULT_DIR = INPUT_DIR / "result"
    RESULT_DIR.mkdir(parents=True, exist_ok=True)
    OUT_PATH = RESULT_DIR / "order_Total_20230524.xlsx"

    VEG_COL = {
        "トマト": 2,
        "キャベツ": 3,
        "レタス": 4,
        "白菜": 5,
        "ほうれん草": 6,
        "大根": 7,
        "ニンジン": 8,   # H列
    }

    HEADERS = [
        "ソース名",
        "トマト",
        "キャベツ",
        "レタス",
        "白菜",
        "ほうれん草",
        "大根",
        "ニンジン",
    ]

    targets = [
        f for f in sorted(INPUT_DIR.glob("*.xlsx"))
        if not f.name.startswith("~$")
    ]
    if not targets:
        raise FileNotFoundError(f"処理対象の Excel ファイルがありません: {INPUT_DIR}")

    print("[Part1] 対象ファイル:")
    for f in targets:
        print(" -", f.name)

    records = []
    for fp in targets:
        wb = load_workbook(fp, data_only=True)
        ws = wb.worksheets[0]

        rec = {"source": fp.name}

        for c in range(1, ws.max_column + 1):
            header = normalize_header(ws.cell(row=1, column=c).value)
            if header in VEG_COL:
                qty = safe_float(ws.cell(row=2, column=c).value)
                rec[header] = qty

        records.append(rec)
        wb.close()

    # ② 集計ファイル作成（毎回新規）
    out_wb = Workbook()
    out_ws = out_wb.active
    out_ws.title = "Total"

    for c, h in enumerate(HEADERS, start=1):
        out_ws.cell(row=1, column=c, value=h)

    for r, rec in enumerate(records, start=2):
        out_ws.cell(row=r, column=1, value=rec["source"])
        for veg, col in VEG_COL.items():
            out_ws.cell(row=r, column=col, value=rec.get(veg))

    out_wb.save(OUT_PATH)
    out_wb.close()

    # ③ TOTAL 行（列合計）
    sum_wb = load_workbook(OUT_PATH)
    ws = sum_wb["Total"]

    last_row = ws.max_row
    last_col = ws.max_column

    ws.cell(row=last_row + 1, column=1, value="TOTAL")

    for c in range(2, last_col + 1):
        total = 0.0
        has_value = False
        for r in range(2, last_row + 1):
            v = safe_float(ws.cell(row=r, column=c).value)
            if v is not None:
                total += v
                has_value = True
        if has_value:
            ws.cell(row=last_row + 1, column=c, value=total)

    sum_wb.save(OUT_PATH)
    sum_wb.close()

    print("[Part1] 完了。出力先:", OUT_PATH)
    return OUT_PATH


# =========================================================
# Part2: inventory_analize_20230524.xlsx 作成
#   入力: /content/order/order_Total_20230524.xlsx, /content/inventory/inventory.xlsx
#   出力: /content/inventory/result/inventory_analyze_20230524.xlsx
# =========================================================
def run_part2_make_inventory_analyze():
    ORDER_PATH = Path("/content/order/result/order_Total_20230524.xlsx")
    INVENTORY_PATH = Path("/content/inventory/inventory.xlsx")

    OUT_DIR = Path("/content/inventory/result")
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    OUT_PATH = OUT_DIR / "inventory_analyze_20230524.xlsx"

    if not ORDER_PATH.exists():
        raise FileNotFoundError(f"order_Total_20230524.xlsx が見つかりません: {ORDER_PATH}")
    if not INVENTORY_PATH.exists():
        raise FileNotFoundError(f"inventory.xlsx が見つかりません: {INVENTORY_PATH}")

    print("[Part2] order file    :", ORDER_PATH)
    print("[Part2] inventory file:", INVENTORY_PATH)

    # ① order_Total の 1行目・最終行目
    order_wb = load_workbook(ORDER_PATH, data_only=True)
    order_ws = order_wb.worksheets[0]
    order_header = get_row(order_ws, 1, order_ws.max_column)
    order_last = get_row(order_ws, order_ws.max_row, order_ws.max_column)
    order_wb.close()

    # ② inventory の 最終行目
    inv_wb = load_workbook(INVENTORY_PATH, data_only=True)
    inv_ws = inv_wb.worksheets[0]
    inv_header = get_row(inv_ws, 1, inv_ws.max_column)
    inv_last = get_row(inv_ws, inv_ws.max_row, inv_ws.max_column)
    inv_wb.close()

    inv_map = {}
    for h, v in zip(inv_header, inv_last):
        key = normalize_header(h)
        if key:
            inv_map[key] = v

    # ③ 新規作成
    out_wb = Workbook()
    out_ws = out_wb.active
    out_ws.title = "Analyze"

    for c, v in enumerate(order_header, start=1):
        out_ws.cell(row=1, column=c, value=v)

    out_ws.cell(row=2, column=1, value="order_total_last")
    for c in range(2, len(order_header) + 1):
        out_ws.cell(row=2, column=c, value=order_last[c - 1])

    out_ws.cell(row=3, column=1, value="inventory_last")
    for c in range(2, len(order_header) + 1):
        col_name = normalize_header(order_header[c - 1])
        out_ws.cell(row=3, column=c, value=inv_map.get(col_name))

    # ④ 差分（3-2）を4行目
    out_ws.cell(row=4, column=1, value="差分(3-2)")
    for c in range(2, len(order_header) + 1):
        v2 = to_number(out_ws.cell(row=2, column=c).value)
        v3 = to_number(out_ws.cell(row=3, column=c).value)
        out_ws.cell(row=4, column=c, value=(v3 - v2) if v2 is not None and v3 is not None else None)

    out_wb.save(OUT_PATH)
    out_wb.close()

    print("[Part2] 完了。出力先:", OUT_PATH)
    return OUT_PATH


# =========================================================
# Part3: inventory_pickup_20230524.xlsx 作成
#   入力: /content/inventory/result/inventory_analyze_20230524.xlsx, /content/inventory/pickup.xlsx
#   出力: /content/inventory/result/inventory_pickup_20230524.xlsx
#   追加: A2=最新在庫数量, A5=発注判定, 5行目=発注判定は、最新在庫数量としきい値の差分(2行目-3行目)
# =========================================================
def run_part3_make_inventory_pickup():
    ANALYZE_PATH = Path("/content/inventory/result/inventory_analyze_20230524.xlsx")
    PICKUP_PATH  = Path("/content/inventory/pickup.xlsx")

    OUT_DIR  = Path("/content/inventory/result")
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    OUT_PATH = OUT_DIR / "inventory_pickup_20230524.xlsx"

    if not ANALYZE_PATH.exists():
        raise FileNotFoundError(f"inventory_analyze_20230524.xlsx が見つかりません: {ANALYZE_PATH}")
    if not PICKUP_PATH.exists():
        raise FileNotFoundError(f"pickup.xlsx が見つかりません: {PICKUP_PATH}")

    print("[Part3] analyze file:", ANALYZE_PATH)
    print("[Part3] pickup file :", PICKUP_PATH)

    # ① analyze 最終行 + ヘッダ
    an_wb = load_workbook(ANALYZE_PATH, data_only=True)
    an_ws = an_wb.worksheets[0]
    an_header = get_row(an_ws, 1, an_ws.max_column)
    an_last   = get_row(an_ws, an_ws.max_row, an_ws.max_column)
    an_wb.close()

    # ② pickup 2行目・3行目
    pk_wb = load_workbook(PICKUP_PATH, data_only=True)
    pk_ws = pk_wb.worksheets[0]
    pk_row2 = get_row(pk_ws, 2, pk_ws.max_column)
    pk_row3 = get_row(pk_ws, 3, pk_ws.max_column)
    pk_wb.close()

    max_cols = max(len(an_header), len(an_last), len(pk_row2), len(pk_row3))
    an_header = pad(an_header, max_cols)
    an_last   = pad(an_last, max_cols)
    pk_row2   = pad(pk_row2, max_cols)
    pk_row3   = pad(pk_row3, max_cols)

    # ③ 新規作成
    out_wb = Workbook()
    out_ws = out_wb.active
    out_ws.title = "Pickup"

    for c, v in enumerate(an_header, start=1):
        out_ws.cell(row=1, column=c, value=v)
    for c, v in enumerate(an_last, start=1):
        out_ws.cell(row=2, column=c, value=v)
    for c, v in enumerate(pk_row2, start=1):
        out_ws.cell(row=3, column=c, value=v)
    for c, v in enumerate(pk_row3, start=1):
        out_ws.cell(row=4, column=c, value=v)

    out_wb.save(OUT_PATH)
    out_wb.close()

    # ④ 差分 + ラベル更新
    wb = load_workbook(OUT_PATH)
    ws = wb["Pickup"]

    ws.cell(row=5, column=1, value="DIFF(2-3)")
    for c in range(2, max_cols + 1):
        v2 = to_number(ws.cell(row=2, column=c).value)
        v3 = to_number(ws.cell(row=3, column=c).value)
        ws.cell(row=5, column=c, value=(v2 - v3) if v2 is not None and v3 is not None else None)

    ws["A2"] = "最新在庫数量"
    ws["A5"] = "発注判定"

    wb.save(OUT_PATH)
    wb.close()

    print("[Part3] 完了。出力先:", OUT_PATH)
    return OUT_PATH


# =========================================================
# Part4: 発注対象抽出 + Mailtrapでメール送信（直書き認証情報）
#   入力: /content/inventory/result/inventory_pickup_20230524.xlsx
#   抽出: 発注判定<=0 の野菜
#   数量: 追加量 行の値
# =========================================================
def read_row_map(ws, label_text):
    max_row = ws.max_row
    max_col = ws.max_column
    headers = [ws.cell(row=1, column=c).value for c in range(1, max_col + 1)]

    target_row = None
    for r in range(2, max_row + 1):
        a = ws.cell(row=r, column=1).value
        if norm_label(a) == norm_label(label_text):
            target_row = r
            break
    if target_row is None:
        raise ValueError(f"行ラベル '{label_text}' がA列に見つかりません。")

    row_values = [ws.cell(row=target_row, column=c).value for c in range(1, max_col + 1)]

    data = {}
    for c in range(2, max_col + 1):
        key = headers[c - 1]
        if key is None or str(key).strip() == "":
            continue
        data[str(key).strip()] = row_values[c - 1]
    return data

def build_order_list(xlsx_path: Path):
    if not xlsx_path.exists():
        raise FileNotFoundError(f"ファイルが見つかりません: {xlsx_path}")

    wb = load_workbook(xlsx_path, data_only=True)
    ws = wb.worksheets[0]

    judgement_map = read_row_map(ws, "発注判定")  # 野菜 -> 判定値
    addqty_map    = read_row_map(ws, "追加量")    # 野菜 -> 追加量

    wb.close()

    order_items = []
    for veg, jv in judgement_map.items():
        j = to_number(jv)
        if j is None:
            continue
        if j <= 0:
            qty_raw = addqty_map.get(veg)
            qty = to_number(qty_raw)
            if qty is None:
                raise ValueError(f"'{veg}' の追加量が数値として取得できません（追加量セル={qty_raw!r}）。")
            order_items.append((veg, qty))
    return order_items

def render_email(subject_date: str, supplier_name: str, order_items):
    lines = [f"・{veg}：{qty:g}" for veg, qty in order_items]
    items_block = "\n".join(lines) if lines else "（発注対象なし）"

    subject = f"【発注依頼】{subject_date}分 野菜の追加発注"
    body = f"""{supplier_name} ご担当者様

お世話になっております。
在庫状況を確認したところ、下記の通り追加発注をお願いいたします。

■発注内容（追加量）
{items_block}

■納品希望
・可能であれば最短便でお願いいたします。
・納品日が難しい場合は、最短の納品予定日をご返信ください。

以上、よろしくお願いいたします。
"""
    return subject, body

def send_via_mailtrap(host, port, username, password, sender, receiver, subject, body):
    msg = EmailMessage()
    msg["From"] = sender
    msg["To"] = receiver
    msg["Subject"] = subject
    msg.set_content(body)

    with smtplib.SMTP(host, port) as server:
        server.ehlo()
        try:
            server.starttls()
            server.ehlo()
        except smtplib.SMTPException:
            pass

        server.login(username, password)
        server.send_message(msg)

def run_part4_send_order_email():
    # ===== Mailtrap（直書き）=====
    username = '**************'
    password = '**************'
    sender_address = 'from@example.com'
    receiver_address = 'to@example.com'

    # Mailtrap SMTP設定（Mailtrap画面の値に合わせる）
    smtp_host = "sandbox.smtp.mailtrap.io"
    smtp_port = 587                # 例：587(TLS) / 2525 / 25 など

    FILE_PATH = Path("/content/inventory/result/inventory_pickup_20230524.xlsx")
    order_items = build_order_list(FILE_PATH)

    supplier_name = "あおぞら農園"
    today = datetime.now().strftime("%Y/%m/%d")
    subject, body = render_email(today, supplier_name, order_items)

    print("[Part4] ==== 発注対象 ====")
    if order_items:
        for veg, qty in order_items:
            print(f"{veg}: {qty:g}")
    else:
        print("発注対象なし")

    print("\n[Part4] ==== 件名 ====")
    print(subject)
    print("\n[Part4] ==== 本文 ====")
    print(body)

    # 発注対象がない場合は送信しない（事故防止）
    if not order_items:
        print("\n[Part4] 送信スキップ：発注対象なし")
        return {"order_items": order_items, "subject": subject, "body": body, "sent": False}

    send_via_mailtrap(
        host=smtp_host,
        port=smtp_port,
        username=username,
        password=password,
        sender=sender_address,
        receiver=receiver_address,
        subject=subject,
        body=body,
    )
    print("\n[Part4] 送信完了（Mailtrapに投函）")
    return {"order_items": order_items, "subject": subject, "body": body, "sent": True}


# =========================================================
# main: 順番実行
# =========================================================
def main():
    print("=== Pipeline Start ===")
    out1 = run_part1_make_order_total()
    out2 = run_part2_make_inventory_analyze()
    out3 = run_part3_make_inventory_pickup()
    out4 = run_part4_send_order_email()
    print("=== Pipeline Done ===")

    return {
        "order_total": out1,
        "inventory_analyze": out2,
        "inventory_pickup": out3,
        "email": out4,
    }

results = main()
results


=== Pipeline Start ===
[Part1] 対象ファイル:
 - order_A_20230524.xlsx
 - order_B_20230524.xlsx
 - order_C_20230524.xlsx
 - order_D_20230524.xlsx
[Part1] 完了。出力先: /content/order/result/order_Total_20230524.xlsx
[Part2] order file    : /content/order/result/order_Total_20230524.xlsx
[Part2] inventory file: /content/inventory/inventory.xlsx
[Part2] 完了。出力先: /content/inventory/result/inventory_analyze_20230524.xlsx
[Part3] analyze file: /content/inventory/result/inventory_analyze_20230524.xlsx
[Part3] pickup file : /content/inventory/pickup.xlsx
[Part3] 完了。出力先: /content/inventory/result/inventory_pickup_20230524.xlsx
[Part4] ==== 発注対象 ====
ニンジン: 80

[Part4] ==== 件名 ====
【発注依頼】2026/02/03分 野菜の追加発注

[Part4] ==== 本文 ====
あおぞら農園 ご担当者様

お世話になっております。
在庫状況を確認したところ、下記の通り追加発注をお願いいたします。

■発注内容（追加量）
・ニンジン：80

■納品希望
・可能であれば最短便でお願いいたします。
・納品日が難しい場合は、最短の納品予定日をご返信ください。

以上、よろしくお願いいたします。


[Part4] 送信完了（Mailtrapに投函）
=== Pipeline Done ===


{'order_total': PosixPath('/content/order/result/order_Total_20230524.xlsx'),
 'inventory_analyze': PosixPath('/content/inventory/result/inventory_analyze_20230524.xlsx'),
 'inventory_pickup': PosixPath('/content/inventory/result/inventory_pickup_20230524.xlsx'),
 'email': {'order_items': [('ニンジン', 80.0)],
  'subject': '【発注依頼】2026/02/03分 野菜の追加発注',
  'body': 'あおぞら農園 ご担当者様\n\nお世話になっております。\n在庫状況を確認したところ、下記の通り追加発注をお願いいたします。\n\n■発注内容（追加量）\n・ニンジン：80\n\n■納品希望\n・可能であれば最短便でお願いいたします。\n・納品日が難しい場合は、最短の納品予定日をご返信ください。\n\n以上、よろしくお願いいたします。\n',
  'sent': True}}

In [ ]:
from pathlib import Path
from openpyxl import load_workbook
from datetime import datetime
from zoneinfo import ZoneInfo
from openpyxl.styles import Alignment
import re

# =========================
# パス設定（固定）
# =========================
PICKUP_PATH = Path("/content/inventory/result/inventory_pickup_20230524.xlsx")
INVENTORY_PATH = Path("/content/inventory/inventory.xlsx")

if not PICKUP_PATH.exists():
    raise FileNotFoundError(f"inventory_pickup_20230524.xlsx が見つかりません: {PICKUP_PATH}")
if not INVENTORY_PATH.exists():
    raise FileNotFoundError(f"inventory.xlsx が見つかりません: {INVENTORY_PATH}")

# =========================
# ユーティリティ
# =========================
def to_number(v):
    if v is None:
        return None
    if isinstance(v, (int, float)):
        return float(v)
    if isinstance(v, str):
        s = re.sub(r"[^0-9\.\-]", "", v.strip())
        if s in ("", "-", ".", "-."):
            return None
        try:
            return float(s)
        except:
            return None
    return None

# =========================
# JSTで現在時刻取得
# =========================
now = datetime.now(ZoneInfo("Asia/Tokyo"))
# 日付はJST、時刻は固定で 0:00:00
timestamp = now.strftime("%Y-%m-%d") + "  0:00:00"
dow_en = now.strftime("%a")  # Mon, Tue, Wed, Thu, Fri, Sat, Sun

# =========================
# ①～④ inventory_pickup.xlsx 更新
# =========================
wb = load_workbook(PICKUP_PATH)
ws = wb.worksheets[0]

# 行番号（仕様固定）
ROW_LATEST = 2   # 最新在庫数量
ROW_ADDQTY = 4   # 追加量
ROW_JUDGE  = 5   # 発注判定
ROW_FINAL  = 6   # 最終在庫

# ① A6 に「最終在庫」
ws.cell(row=ROW_FINAL, column=1, value="最終在庫")

final_values = []

# 対象列：B～H（=2～8）
for col in range(2, 9):
    latest = to_number(ws.cell(row=ROW_LATEST, column=col).value) or 0.0
    addqty = to_number(ws.cell(row=ROW_ADDQTY, column=col).value) or 0.0
    judge  = to_number(ws.cell(row=ROW_JUDGE,  column=col).value)

    # ②③ 最終在庫計算
    if judge is None or judge > 0:
        final = latest
    else:
        final = latest + addqty

    ws.cell(row=ROW_FINAL, column=col, value=final)
    final_values.append(final)

wb.save(PICKUP_PATH)
wb.close()

print("[Part5] pickup 更新完了:", PICKUP_PATH)
print("[Part5] 最終在庫（B6:H6）:", final_values)

# =========================
# ⑤ inventory.xlsx に履歴追記
# =========================
inv_wb = load_workbook(INVENTORY_PATH)
inv_ws = inv_wb.worksheets[0]

new_row = inv_ws.max_row + 1

# A列：日時（JST） / B列：曜日（英語短縮）
inv_ws.cell(row=new_row, column=1, value=timestamp)
inv_ws.cell(row=new_row, column=2, value=dow_en)

# A列：日付・時間（右寄せ）
cell_a = inv_ws.cell(row=new_row, column=1, value=timestamp)
cell_a.alignment = Alignment(horizontal="right")

# C～I列：最終在庫（B6～H6）
for i, v in enumerate(final_values, start=3):  # C=3 .. I=9
    inv_ws.cell(row=new_row, column=i, value=v)

inv_wb.save(INVENTORY_PATH)
inv_wb.close()

print("[Part5] inventory 履歴追記完了:", INVENTORY_PATH)
print(f"[Part5] 追記行: {new_row} / {timestamp} ({dow_en})")


[Part5] pickup 更新完了: /content/inventory/result/inventory_pickup_20230524.xlsx
[Part5] 最終在庫（B6:H6）: [60.0, 52.0, 61.0, 59.0, 52.0, 33.0, 98.0]
[Part5] inventory 履歴追記完了: /content/inventory/inventory.xlsx
[Part5] 追記行: 19 / 2026-02-03  0:00:00 (Tue)


In [ ]:
from pathlib import Path
import re
from datetime import datetime
from zoneinfo import ZoneInfo
from openpyxl import load_workbook, Workbook
from openpyxl.styles import Alignment
import smtplib
from email.message import EmailMessage

# =========================================================
# 共通ユーティリティ
# =========================================================
def normalize_header(v):
    """見出しの表記ゆれ対策"""
    if v is None:
        return ""
    s = str(v).strip()
    s = s.replace("　", "")                 # 全角スペース除去
    s = re.sub(r"\(.*?\)$", "", s)          # (kg) 等除去
    return s

def to_number(v):
    """数値 or 数値っぽい文字列 -> float / 不可なら None"""
    if v is None:
        return None
    if isinstance(v, (int, float)):
        return float(v)
    if isinstance(v, str):
        s = re.sub(r"[^0-9\.\-]", "", v.strip())
        if s in ("", "-", ".", "-."):
            return None
        try:
            return float(s)
        except:
            return None
    return None

def safe_float(v):
    """（Part1互換）数値 or 数値っぽい文字列を float に"""
    return to_number(v)

def get_row(ws, row_idx, max_col):
    return [ws.cell(row=row_idx, column=c).value for c in range(1, max_col + 1)]

def pad(row, n):
    return row + [None] * (n - len(row)) if len(row) < n else row[:n]

def norm_label(x):
    """A列ラベル照合用：前後空白・全角スペース除去"""
    if x is None:
        return ""
    return str(x).strip().replace("　", "")


# =========================================================
# Part1: order_Total_20230524.xlsx 作成
#   入力: /content/order/*.xlsx
#   出力: /content/order/result/order_Total_20230524.xlsx
# =========================================================
def run_part1_make_order_total():
    INPUT_DIR = Path("/content/drive/MyDrive/samples/order_new")
    RESULT_DIR = INPUT_DIR / "result"
    RESULT_DIR.mkdir(parents=True, exist_ok=True)
    OUT_PATH = RESULT_DIR / "order_Total_20230524.xlsx"

    VEG_COL = {
        "トマト": 2,
        "キャベツ": 3,
        "レタス": 4,
        "白菜": 5,
        "ほうれん草": 6,
        "大根": 7,
        "ニンジン": 8,   # H列
    }

    HEADERS = [
        "ソース名",
        "トマト",
        "キャベツ",
        "レタス",
        "白菜",
        "ほうれん草",
        "大根",
        "ニンジン",
    ]

    targets = [
        f for f in sorted(INPUT_DIR.glob("*.xlsx"))
        if not f.name.startswith("~$")
    ]
    if not targets:
        raise FileNotFoundError(f"処理対象の Excel ファイルがありません: {INPUT_DIR}")

    print("[Part1] 対象ファイル:")
    for f in targets:
        print(" -", f.name)

    records = []
    for fp in targets:
        wb = load_workbook(fp, data_only=True)
        ws = wb.worksheets[0]

        rec = {"source": fp.name}

        for c in range(1, ws.max_column + 1):
            header = normalize_header(ws.cell(row=1, column=c).value)
            if header in VEG_COL:
                qty = safe_float(ws.cell(row=2, column=c).value)
                rec[header] = qty

        records.append(rec)
        wb.close()

    # ② 集計ファイル作成（毎回新規）
    out_wb = Workbook()
    out_ws = out_wb.active
    out_ws.title = "Total"

    for c, h in enumerate(HEADERS, start=1):
        out_ws.cell(row=1, column=c, value=h)

    for r, rec in enumerate(records, start=2):
        out_ws.cell(row=r, column=1, value=rec["source"])
        for veg, col in VEG_COL.items():
            out_ws.cell(row=r, column=col, value=rec.get(veg))

    out_wb.save(OUT_PATH)
    out_wb.close()

    # ③ TOTAL 行（列合計）
    sum_wb = load_workbook(OUT_PATH)
    ws = sum_wb["Total"]

    last_row = ws.max_row
    last_col = ws.max_column

    ws.cell(row=last_row + 1, column=1, value="TOTAL")

    for c in range(2, last_col + 1):
        total = 0.0
        has_value = False
        for r in range(2, last_row + 1):
            v = safe_float(ws.cell(row=r, column=c).value)
            if v is not None:
                total += v
                has_value = True
        if has_value:
            ws.cell(row=last_row + 1, column=c, value=total)

    sum_wb.save(OUT_PATH)
    sum_wb.close()

    print("[Part1] 完了。出力先:", OUT_PATH)
    return OUT_PATH


# =========================================================
# Part2: inventory_analyze_20230524.xlsx 作成
#   入力: /content/order/result/order_Total_20230524.xlsx, /content/inventory/inventory.xlsx
#   出力: /content/inventory/result/inventory_analyze_20230524.xlsx
# =========================================================
def run_part2_make_inventory_analyze():
    ORDER_PATH = Path("/content/drive/MyDrive/samples/order_new/result/order_Total_20230524.xlsx")
    INVENTORY_PATH = Path("/content/drive/MyDrive/samples/inventory.xlsx")

    OUT_DIR = Path("/content/drive/MyDrive/samples/result")
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    OUT_PATH = OUT_DIR / "inventory_analyze_20230524.xlsx"

    if not ORDER_PATH.exists():
        raise FileNotFoundError(f"order_Total_20230524.xlsx が見つかりません: {ORDER_PATH}")
    if not INVENTORY_PATH.exists():
        raise FileNotFoundError(f"inventory.xlsx が見つかりません: {INVENTORY_PATH}")

    print("[Part2] order file    :", ORDER_PATH)
    print("[Part2] inventory file:", INVENTORY_PATH)

    # ① order_Total の 1行目・最終行目
    order_wb = load_workbook(ORDER_PATH, data_only=True)
    order_ws = order_wb.worksheets[0]
    order_header = get_row(order_ws, 1, order_ws.max_column)
    order_last = get_row(order_ws, order_ws.max_row, order_ws.max_column)
    order_wb.close()

    # ② inventory の 最終行目
    inv_wb = load_workbook(INVENTORY_PATH, data_only=True)
    inv_ws = inv_wb.worksheets[0]
    inv_header = get_row(inv_ws, 1, inv_ws.max_column)
    inv_last = get_row(inv_ws, inv_ws.max_row, inv_ws.max_column)
    inv_wb.close()

    inv_map = {}
    for h, v in zip(inv_header, inv_last):
        key = normalize_header(h)
        if key:
            inv_map[key] = v

    # ③ 新規作成
    out_wb = Workbook()
    out_ws = out_wb.active
    out_ws.title = "Analyze"

    for c, v in enumerate(order_header, start=1):
        out_ws.cell(row=1, column=c, value=v)

    out_ws.cell(row=2, column=1, value="order_total_last")
    for c in range(2, len(order_header) + 1):
        out_ws.cell(row=2, column=c, value=order_last[c - 1])

    out_ws.cell(row=3, column=1, value="inventory_last")
    for c in range(2, len(order_header) + 1):
        col_name = normalize_header(order_header[c - 1])
        out_ws.cell(row=3, column=c, value=inv_map.get(col_name))

    # ④ 差分（3-2）を4行目
    out_ws.cell(row=4, column=1, value="差分(3-2)")
    for c in range(2, len(order_header) + 1):
        v2 = to_number(out_ws.cell(row=2, column=c).value)
        v3 = to_number(out_ws.cell(row=3, column=c).value)
        out_ws.cell(row=4, column=c, value=(v3 - v2) if v2 is not None and v3 is not None else None)

    out_wb.save(OUT_PATH)
    out_wb.close()

    print("[Part2] 完了。出力先:", OUT_PATH)
    return OUT_PATH


# =========================================================
# Part3: inventory_pickup_20230524.xlsx 作成
#   入力: /content/inventory/result/inventory_analyze_20230524.xlsx, /content/inventory/pickup.xlsx
#   出力: /content/inventory/result/inventory_pickup_20230524.xlsx
#   追加: A2=最新在庫数量, A5=発注判定, 5行目=発注判定は、最新在庫数量としきい値の差分(2行目-3行目)
# =========================================================
def run_part3_make_inventory_pickup():
    ANALYZE_PATH = Path("/content/drive/MyDrive/samples/result/inventory_analyze_20230524.xlsx")
    PICKUP_PATH  = Path("/content/drive/MyDrive/samples/pickup.xlsx")

    OUT_DIR  = Path("/content/drive/MyDrive/samples/result")
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    OUT_PATH = OUT_DIR / "inventory_pickup_20230524.xlsx"

    if not ANALYZE_PATH.exists():
        raise FileNotFoundError(f"inventory_analyze_20230524.xlsx が見つかりません: {ANALYZE_PATH}")
    if not PICKUP_PATH.exists():
        raise FileNotFoundError(f"pickup.xlsx が見つかりません: {PICKUP_PATH}")

    print("[Part3] analyze file:", ANALYZE_PATH)
    print("[Part3] pickup file :", PICKUP_PATH)

    # ① analyze 最終行 + ヘッダ
    an_wb = load_workbook(ANALYZE_PATH, data_only=True)
    an_ws = an_wb.worksheets[0]
    an_header = get_row(an_ws, 1, an_ws.max_column)
    an_last   = get_row(an_ws, an_ws.max_row, an_ws.max_column)
    an_wb.close()

    # ② pickup 2行目・3行目
    pk_wb = load_workbook(PICKUP_PATH, data_only=True)
    pk_ws = pk_wb.worksheets[0]
    pk_row2 = get_row(pk_ws, 2, pk_ws.max_column)
    pk_row3 = get_row(pk_ws, 3, pk_ws.max_column)
    pk_wb.close()

    max_cols = max(len(an_header), len(an_last), len(pk_row2), len(pk_row3))
    an_header = pad(an_header, max_cols)
    an_last   = pad(an_last, max_cols)
    pk_row2   = pad(pk_row2, max_cols)
    pk_row3   = pad(pk_row3, max_cols)

    # ③ 新規作成
    out_wb = Workbook()
    out_ws = out_wb.active
    out_ws.title = "Pickup"

    for c, v in enumerate(an_header, start=1):
        out_ws.cell(row=1, column=c, value=v)
    for c, v in enumerate(an_last, start=1):
        out_ws.cell(row=2, column=c, value=v)
    for c, v in enumerate(pk_row2, start=1):
        out_ws.cell(row=3, column=c, value=v)
    for c, v in enumerate(pk_row3, start=1):
        out_ws.cell(row=4, column=c, value=v)

    out_wb.save(OUT_PATH)
    out_wb.close()

    # ④ 差分 + ラベル更新
    wb = load_workbook(OUT_PATH)
    ws = wb["Pickup"]

    ws.cell(row=5, column=1, value="DIFF(2-3)")
    for c in range(2, max_cols + 1):
        v2 = to_number(ws.cell(row=2, column=c).value)
        v3 = to_number(ws.cell(row=3, column=c).value)
        ws.cell(row=5, column=c, value=(v2 - v3) if v2 is not None and v3 is not None else None)

    ws["A2"] = "最新在庫数量"
    ws["A5"] = "発注判定"

    wb.save(OUT_PATH)
    wb.close()

    print("[Part3] 完了。出力先:", OUT_PATH)
    return OUT_PATH


# =========================================================
# Part4: 発注対象抽出 + Mailtrapでメール送信（直書き認証情報）
#   入力: /content/inventory/result/inventory_pickup_20230524.xlsx
#   抽出: 発注判定<=0 の野菜
#   数量: 追加量 行の値
# =========================================================
def read_row_map(ws, label_text):
    max_row = ws.max_row
    max_col = ws.max_column
    headers = [ws.cell(row=1, column=c).value for c in range(1, max_col + 1)]

    target_row = None
    for r in range(2, max_row + 1):
        a = ws.cell(row=r, column=1).value
        if norm_label(a) == norm_label(label_text):
            target_row = r
            break
    if target_row is None:
        raise ValueError(f"行ラベル '{label_text}' がA列に見つかりません。")

    row_values = [ws.cell(row=target_row, column=c).value for c in range(1, max_col + 1)]

    data = {}
    for c in range(2, max_col + 1):
        key = headers[c - 1]
        if key is None or str(key).strip() == "":
            continue
        data[str(key).strip()] = row_values[c - 1]
    return data

def build_order_list(xlsx_path: Path):
    if not xlsx_path.exists():
        raise FileNotFoundError(f"ファイルが見つかりません: {xlsx_path}")

    wb = load_workbook(xlsx_path, data_only=True)
    ws = wb.worksheets[0]

    judgement_map = read_row_map(ws, "発注判定")  # 野菜 -> 判定値
    addqty_map    = read_row_map(ws, "追加量")    # 野菜 -> 追加量

    wb.close()

    order_items = []
    for veg, jv in judgement_map.items():
        j = to_number(jv)
        if j is None:
            continue
        if j <= 0:
            qty_raw = addqty_map.get(veg)
            qty = to_number(qty_raw)
            if qty is None:
                raise ValueError(f"'{veg}' の追加量が数値として取得できません（追加量セル={qty_raw!r}）。")
            order_items.append((veg, qty))
    return order_items

def render_email(subject_date: str, supplier_name: str, order_items):
    lines = [f"・{veg}：{qty:g}" for veg, qty in order_items]
    items_block = "\n".join(lines) if lines else "（発注対象なし）"

    subject = f"【発注依頼】{subject_date}分 野菜の追加発注"
    body = f"""{supplier_name} ご担当者様

お世話になっております。
在庫状況を確認したところ、下記の通り追加発注をお願いいたします。

■発注内容（追加量）
{items_block}

■納品希望
・可能であれば最短便でお願いいたします。
・納品日が難しい場合は、最短の納品予定日をご返信ください。

以上、よろしくお願いいたします。
"""
    return subject, body

def send_via_mailtrap(host, port, username, password, sender, receiver, subject, body):
    msg = EmailMessage()
    msg["From"] = sender
    msg["To"] = receiver
    msg["Subject"] = subject
    msg.set_content(body)

    with smtplib.SMTP(host, port) as server:
        server.ehlo()
        try:
            server.starttls()
            server.ehlo()
        except smtplib.SMTPException:
            pass

        server.login(username, password)
        server.send_message(msg)

def run_part4_send_order_email():
    # ===== Mailtrap（直書き）=====
    username = 'e115c13d150ca6'
    password = '101451ae41baa6'
    sender_address = 'from@example.com'
    receiver_address = 'to@example.com'

    # Mailtrap SMTP設定（Mailtrap画面の値に合わせる）
    smtp_host = "sandbox.smtp.mailtrap.io"
    smtp_port = 587

    FILE_PATH = Path("/content/drive/MyDrive/samples/result/inventory_pickup_20230524.xlsx")
    order_items = build_order_list(FILE_PATH)

    supplier_name = "あおぞら農園"
    today = datetime.now().strftime("%Y/%m/%d")
    subject, body = render_email(today, supplier_name, order_items)

    print("[Part4] ==== 発注対象 ====")
    if order_items:
        for veg, qty in order_items:
            print(f"{veg}: {qty:g}")
    else:
        print("発注対象なし")

    print("\n[Part4] ==== 件名 ====")
    print(subject)
    print("\n[Part4] ==== 本文 ====")
    print(body)

    if not order_items:
        print("\n[Part4] 送信スキップ：発注対象なし")
        return {"order_items": order_items, "subject": subject, "body": body, "sent": False}

    send_via_mailtrap(
        host=smtp_host,
        port=smtp_port,
        username=username,
        password=password,
        sender=sender_address,
        receiver=receiver_address,
        subject=subject,
        body=body,
    )
    print("\n[Part4] 送信完了（Mailtrapに投函）")
    return {"order_items": order_items, "subject": subject, "body": body, "sent": True}


# =========================================================
# Part5: 最終在庫計算 + inventory.xlsx へ履歴追記
#   入力: /content/inventory/result/inventory_pickup_20230524.xlsx
#   更新: 6行目(A6)に最終在庫を作成（B6:H6）
#   追記: /content/inventory/inventory.xlsx の末尾に履歴追記
#         A列=日付時間(右寄せ), B列=曜日(英語短縮), C..I=最終在庫
# =========================================================
def run_part5_update_inventory_history():
    PICKUP_PATH = Path("/content/drive/MyDrive/samples/result/inventory_pickup_20230524.xlsx")
    INVENTORY_PATH = Path("/content/drive/MyDrive/samples/inventory.xlsx")

    if not PICKUP_PATH.exists():
        raise FileNotFoundError(f"inventory_pickup_20230524.xlsx が見つかりません: {PICKUP_PATH}")
    if not INVENTORY_PATH.exists():
        raise FileNotFoundError(f"inventory.xlsx が見つかりません: {INVENTORY_PATH}")

    # JSTで現在時刻取得
    now = datetime.now(ZoneInfo("Asia/Tokyo"))
    # ★ユーザー指定どおり：YYYY-MM-DD HH:MM:SS
    timestamp = now.strftime("%Y-%m-%d %H:%M:%S")
    dow_en = now.strftime("%a")  # Mon, Tue, Wed, Thu, Fri, Sat, Sun

    # ①～④ inventory_pickup.xlsx 更新
    wb = load_workbook(PICKUP_PATH)
    ws = wb.worksheets[0]

    ROW_LATEST = 2   # 最新在庫数量
    ROW_ADDQTY = 4   # 追加量
    ROW_JUDGE  = 5   # 発注判定
    ROW_FINAL  = 6   # 最終在庫

    ws.cell(row=ROW_FINAL, column=1, value="最終在庫")

    final_values = []
    for col in range(2, 9):  # B..H
        latest = to_number(ws.cell(row=ROW_LATEST, column=col).value) or 0.0
        addqty = to_number(ws.cell(row=ROW_ADDQTY, column=col).value) or 0.0
        judge  = to_number(ws.cell(row=ROW_JUDGE,  column=col).value)

        if judge is None or judge > 0:
            final = latest
        else:
            final = latest + addqty

        ws.cell(row=ROW_FINAL, column=col, value=final)
        final_values.append(final)

    wb.save(PICKUP_PATH)
    wb.close()

    print("[Part5] pickup 更新完了:", PICKUP_PATH)
    print("[Part5] 最終在庫（B6:H6）:", final_values)

    # ⑤ inventory.xlsx に履歴追記（末尾）
    inv_wb = load_workbook(INVENTORY_PATH)
    inv_ws = inv_wb.worksheets[0]

    new_row = inv_ws.max_row + 1

    # A列：日時（右寄せ）
    cell_a = inv_ws.cell(row=new_row, column=1, value=timestamp)
    cell_a.alignment = Alignment(horizontal="right")

    # B列：曜日（英語短縮）※必要なら右寄せ/中央寄せなど変更可
    inv_ws.cell(row=new_row, column=2, value=dow_en)

    # C～I列：最終在庫
    for i, v in enumerate(final_values, start=3):  # C=3 .. I=9
        inv_ws.cell(row=new_row, column=i, value=v)

    inv_wb.save(INVENTORY_PATH)
    inv_wb.close()

    print("[Part5] inventory 履歴追記完了:", INVENTORY_PATH)
    print(f"[Part5] 追記行: {new_row} / {timestamp} ({dow_en})")

    return {"inventory_path": str(INVENTORY_PATH), "appended_row": new_row, "timestamp": timestamp, "dow": dow_en}


# =========================================================
# main: 順番実行
# =========================================================
def main():
    print("=== Pipeline Start ===")
    out1 = run_part1_make_order_total()
    out2 = run_part2_make_inventory_analyze()
    out3 = run_part3_make_inventory_pickup()
    out4 = run_part4_send_order_email()
    out5 = run_part5_update_inventory_history()
    print("=== Pipeline Done ===")

    return {
        "order_total": out1,
        "inventory_analyze": out2,
        "inventory_pickup": out3,
        "email": out4,
        "inventory_history": out5,
    }

results = main()
results


=== Pipeline Start ===
[Part1] 対象ファイル:
 - order_A_20230524.xlsx
 - order_B_20230524.xlsx
 - order_C_20230524.xlsx
 - order_D_20230524.xlsx
[Part1] 完了。出力先: /content/drive/MyDrive/samples/order_new/result/order_Total_20230524.xlsx
[Part2] order file    : /content/drive/MyDrive/samples/order_new/result/order_Total_20230524.xlsx
[Part2] inventory file: /content/drive/MyDrive/samples/inventory.xlsx
[Part2] 完了。出力先: /content/drive/MyDrive/samples/result/inventory_analyze_20230524.xlsx
[Part3] analyze file: /content/drive/MyDrive/samples/result/inventory_analyze_20230524.xlsx
[Part3] pickup file : /content/drive/MyDrive/samples/pickup.xlsx
[Part3] 完了。出力先: /content/drive/MyDrive/samples/result/inventory_pickup_20230524.xlsx
[Part4] ==== 発注対象 ====
トマト: 2
キャベツ: 80
レタス: 100
白菜: 60
ほうれん草: 80
大根: 60
ニンジン: 80

[Part4] ==== 件名 ====
【発注依頼】2026/02/04分 野菜の追加発注

[Part4] ==== 本文 ====
あおぞら農園 ご担当者様

お世話になっております。
在庫状況を確認したところ、下記の通り追加発注をお願いいたします。

■発注内容（追加量）
・トマト：2
・キャベツ：80
・レタス：100
・白菜：60
・ほうれん草：80
・大根：60
・ニン

{'order_total': PosixPath('/content/drive/MyDrive/samples/order_new/result/order_Total_20230524.xlsx'),
 'inventory_analyze': PosixPath('/content/drive/MyDrive/samples/result/inventory_analyze_20230524.xlsx'),
 'inventory_pickup': PosixPath('/content/drive/MyDrive/samples/result/inventory_pickup_20230524.xlsx'),
 'email': {'order_items': [('トマト', 2.0),
   ('キャベツ', 80.0),
   ('レタス', 100.0),
   ('白菜', 60.0),
   ('ほうれん草', 80.0),
   ('大根', 60.0),
   ('ニンジン', 80.0)],
  'subject': '【発注依頼】2026/02/04分 野菜の追加発注',
  'body': 'あおぞら農園 ご担当者様\n\nお世話になっております。\n在庫状況を確認したところ、下記の通り追加発注をお願いいたします。\n\n■発注内容（追加量）\n・トマト：2\n・キャベツ：80\n・レタス：100\n・白菜：60\n・ほうれん草：80\n・大根：60\n・ニンジン：80\n\n■納品希望\n・可能であれば最短便でお願いいたします。\n・納品日が難しい場合は、最短の納品予定日をご返信ください。\n\n以上、よろしくお願いいたします。\n',
  'sent': True},
 'inventory_history': {'inventory_path': '/content/drive/MyDrive/samples/inventory.xlsx',
  'appended_row': 1002,
  'timestamp': '2026-02-04 19:20:43',
  'dow': 'Wed'}}